# Part 4.2 | Weekly retention by signup-week cohort

> Weekly retention by signup-week cohort (share of activated users with a successful transaction in week 1, 2, 4 and 8 after signup).

Run the prep cell, then the SQL cell to reproduce the results locally. The prep cell loads the original CSVs and translates date expressions for SQLite. To run the query against the case database, copy the SQL body without the `%%sql` line.

**Assumptions**

- Include Q3 signups using the case’s fixed Cairo offset of UTC+2. Calendar cohorts start on Sunday, and the first and last cohorts may contain partial signup weeks.
- Activation requires KYC verification by extract and a successful deposit within seven days of signup. Retention counts any successful transaction during the specified week, measured from each user’s signup timestamp. The qualifying deposit can count toward week 1 if it falls inside that window.
- The extract includes November 5 through the end of the UTC day. Each rate includes only activated users whose entire follow-up window was observable. A blank rate means no activated user in that cohort had completed the window.

In [2]:
from pathlib import Path
import re
import sqlite3

import pandas as pd
from IPython import get_ipython
from IPython.display import display

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "raw").is_dir()
)
RAW = ROOT / "data" / "raw"
users = pd.read_csv(next(RAW.glob("* - users.csv")))
transactions = pd.read_csv(next(RAW.glob("* - transactions.csv")))
assert users.drop_duplicates()["user_id"].is_unique

connection = sqlite3.connect(":memory:")
connection.execute("ATTACH DATABASE ':memory:' AS lucky_case")
for name, frame in [("users", users), ("transactions", transactions)]:
    frame.to_sql(name, connection, index=False)
    connection.execute(f"CREATE TABLE lucky_case.{name} AS SELECT * FROM main.{name}")


def run_case_sql(line, cell):
    # Translate the timestamp literals, interval arithmetic and cohort expression.
    local_sql = re.sub(r"\bTIMESTAMP ('[^']+')", r"\1", cell)
    local_sql = local_sql.replace(
        "DATE_TRUNC('week', signup_cairo + INTERVAL '1 day') - INTERVAL '1 day'",
        "DATE(signup_cairo, '-' || STRFTIME('%w', signup_cairo) || ' days')",
    )
    local_sql = re.sub(
        r"(\b\w+(?:\.\w+)?) \+ INTERVAL '(\d+) (days?|hours?)'",
        r"DATETIME(\1, '+\2 \3')",
        local_sql,
    )
    display(pd.read_sql_query(local_sql, connection))


get_ipython().register_magic_function(run_case_sql, magic_kind="cell", magic_name="sql")

In [3]:
%%sql
-- Part 4.2 | Weekly retention by signup-week cohort.

-- Exclude apparent QA accounts, collapse repeated user rows, and calculate Cairo signup time.
WITH clean_users AS (
    SELECT DISTINCT user_id, signup_ts, kyc_verified_ts,
           signup_ts + INTERVAL '2 hours' AS signup_cairo
    FROM lucky_case.users
    WHERE user_id IS NOT NULL
      AND COALESCE(LOWER(TRIM(full_name)), '') NOT IN (
          'qa lucky', 'test user', 'test qa', 'test test',
          'تجربه تطبيق', 'تجربة'
      )
),

-- Keep successful transactions through the extract and collapse exact copies.
successful_transactions AS (
    SELECT DISTINCT txn_id, user_id, txn_ts, txn_type, amount_egp
    FROM lucky_case.transactions
    WHERE status = 'success'
      AND txn_ts < TIMESTAMP '2026-11-06 00:00:00'
),

-- Identify activated Q3 signups and assign each to a Sunday-start Cairo cohort.
activated AS (
    SELECT u.user_id, u.signup_ts,
           DATE_TRUNC('week', signup_cairo + INTERVAL '1 day') - INTERVAL '1 day'
               AS signup_week
    FROM clean_users AS u
    WHERE u.signup_cairo >= TIMESTAMP '2026-07-01 00:00:00'
      AND u.signup_cairo <  TIMESTAMP '2026-10-01 00:00:00'
      AND u.kyc_verified_ts < TIMESTAMP '2026-11-06 00:00:00'
      AND EXISTS (
          SELECT 1
          FROM successful_transactions AS t
          WHERE t.user_id = u.user_id
            AND t.txn_type = 'deposit'
            AND t.txn_ts >= u.signup_ts
            AND t.txn_ts <= u.signup_ts + INTERVAL '7 days'
      )
),

-- Set follow-up windows relative to each user's exact signup time.
-- Week 1 is days [0, 7), week 2 [7, 14), week 4 [21, 28), and week 8 [49, 56).
user_windows AS (
    SELECT *,
           signup_ts + INTERVAL '0 days' AS w1_start,
           signup_ts + INTERVAL '7 days' AS w2_start,
           signup_ts + INTERVAL '14 days' AS w2_end,
           signup_ts + INTERVAL '21 days' AS w4_start,
           signup_ts + INTERVAL '28 days' AS w4_end,
           signup_ts + INTERVAL '49 days' AS w8_start,
           signup_ts + INTERVAL '56 days' AS w8_end
    FROM activated
),

-- Flag completed windows and successful activity once per user per week.
-- The left join preserves activated users with no activity in a follow-up window.
user_flags AS (
    SELECT w.user_id, w.signup_week,
           CASE WHEN w.w2_start <= TIMESTAMP '2026-11-06 00:00:00' THEN 1 ELSE 0 END AS w1_observed,
           CASE WHEN w.w2_end <= TIMESTAMP '2026-11-06 00:00:00' THEN 1 ELSE 0 END AS w2_observed,
           CASE WHEN w.w4_end <= TIMESTAMP '2026-11-06 00:00:00' THEN 1 ELSE 0 END AS w4_observed,
           CASE WHEN w.w8_end <= TIMESTAMP '2026-11-06 00:00:00' THEN 1 ELSE 0 END AS w8_observed,
           MAX(CASE WHEN t.txn_ts >= w.w1_start AND t.txn_ts < w.w2_start THEN 1 ELSE 0 END) AS w1_returned,
           MAX(CASE WHEN t.txn_ts >= w.w2_start AND t.txn_ts < w.w2_end THEN 1 ELSE 0 END) AS w2_returned,
           MAX(CASE WHEN t.txn_ts >= w.w4_start AND t.txn_ts < w.w4_end THEN 1 ELSE 0 END) AS w4_returned,
           MAX(CASE WHEN t.txn_ts >= w.w8_start AND t.txn_ts < w.w8_end THEN 1 ELSE 0 END) AS w8_returned
    FROM user_windows AS w
    LEFT JOIN successful_transactions AS t
        ON t.user_id = w.user_id
       AND t.txn_ts >= w.w1_start
       AND t.txn_ts < w.w8_end
    GROUP BY w.user_id, w.signup_week, w.w2_start, w.w2_end, w.w4_end, w.w8_end
)

-- Report cohort size, observed users, retained users, and retention for each week.
-- A retention rate is NULL when no user has completed that week's window.
SELECT signup_week,
       COUNT(*) AS activated_users,
       SUM(w1_observed) AS w1_observed,
       SUM(w1_returned * w1_observed) AS w1_retained,
       ROUND(100.0 * SUM(w1_returned * w1_observed) / NULLIF(SUM(w1_observed), 0), 2) AS w1_retention_pct,
       SUM(w2_observed) AS w2_observed,
       SUM(w2_returned * w2_observed) AS w2_retained,
       ROUND(100.0 * SUM(w2_returned * w2_observed) / NULLIF(SUM(w2_observed), 0), 2) AS w2_retention_pct,
       SUM(w4_observed) AS w4_observed,
       SUM(w4_returned * w4_observed) AS w4_retained,
       ROUND(100.0 * SUM(w4_returned * w4_observed) / NULLIF(SUM(w4_observed), 0), 2) AS w4_retention_pct,
       SUM(w8_observed) AS w8_observed,
       SUM(w8_returned * w8_observed) AS w8_retained,
       ROUND(100.0 * SUM(w8_returned * w8_observed) / NULLIF(SUM(w8_observed), 0), 2) AS w8_retention_pct
FROM user_flags
GROUP BY signup_week
ORDER BY signup_week;

,signup_week,activated_users,w1_observed,w1_retained,w1_retention_pct,w2_observed,w2_retained,w2_retention_pct,w4_observed,w4_retained,w4_retention_pct,w8_observed,w8_retained,w8_retention_pct
0,2026-06-28,95,95,95,100.0,95,69,72.63,95,64,67.37,95,47,49.47
1,2026-07-05,182,182,182,100.0,182,130,71.43,182,107,58.79,182,85,46.70
2,2026-07-12,163,163,163,100.0,163,114,69.94,163,98,60.12,163,79,48.47
3,2026-07-19,164,164,164,100.0,164,104,63.41,164,98,59.76,164,85,51.83
4,2026-07-26,159,159,159,100.0,159,110,69.18,159,97,61.01,159,80,50.31
5,2026-08-02,181,181,181,100.0,181,122,67.40,181,120,66.30,181,86,47.51
6,2026-08-09,191,191,191,100.0,191,123,64.40,191,112,58.64,191,93,48.69
7,2026-08-16,158,158,158,100.0,158,111,70.25,158,93,58.86,158,75,47.47
8,2026-08-23,179,179,179,100.0,179,123,68.72,179,106,59.22,179,80,44.69
9,2026-08-30,206,206,206,100.0,206,133,64.56,206,128,62.14,206,98,47.57
